# nb03 — CPI aggregation replication (Session 3A, Task 5)

Reconstruct **headline** (All items, `SA0`) and **core** (less food & energy, `SA0L1E`) CPI
from official component indices + published relative importances, using **official inputs only**
(no proxies), then measure the **SA-conversion overhead**. All logic lives in
`src/nowcast/aggregate.py` (package-final; this notebook only imports and displays).
Full write-up: `docs/aggregation_error.md`.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from nowcast import aggregate as A
from nowcast import weights

## Partitions — coarsest complete published set
Each published aggregate already embeds BLS's exact sub-aggregation, so keeping a branch whole
beats re-deriving it from finer rounded RIs. Headline = the 8 major groups; core carves out
food (SAF1) and energy (SETB/SEHE/SEHF).

In [2]:
head_part = A.complete_published_partition('SA0')
core_part = A.complete_published_partition('SA0', (A.FOOD_SUBTREE, *A.ENERGY_SUBTREES))
RI = weights.weights_as_of('2024-06-01')
print('headline:', len(head_part), 'comps, wt=%.1f%%' % sum(RI.get(c,0) for c in head_part), sorted(head_part))
print('core:    ', len(core_part), 'comps, wt=%.1f%%' % sum(RI.get(c,0) for c in core_part), sorted(core_part))

headline: 8 comps, wt=100.0% ['SAA', 'SAE', 'SAF', 'SAG', 'SAH', 'SAM', 'SAR', 'SAT']
core:     15 comps, wt=80.1% ['SAA', 'SAE', 'SAF116', 'SAG', 'SAH1', 'SAH3', 'SAM', 'SAR', 'SEHG', 'SETA', 'SETC', 'SETD', 'SETE', 'SETF', 'SETG']


## Result 1 — NSA reconstruction vs official (≤1 bp target)

In [3]:
rows = []
for agg in ['headline', 'core']:
    r = A.reconstruction_error(agg, seasonal='NSA')
    rows.append({'aggregate': agg, 'components': r['n_components'], 'MAE_2023+': r['mae_2023plus_bp'],
                 'MAE_full': r['mae_bp'], 'median': r['median_bp'], 'max': r['max_bp']})
pd.DataFrame(rows).set_index('aggregate')

,components,MAE_2023+,MAE_full,median,max
aggregate,,,,,
headline,8,0.50,1.83,0.84,10.54
core,15,1.32,1.74,1.14,12.42


In [4]:
# per-year: the break at 2023 is the biennial->annual weight-methodology switch
pd.DataFrame({agg: A.reconstruction_error(agg, seasonal='NSA')['mae_by_year']
              for agg in ['headline', 'core']})

,headline,core
2020,0.90,1.00
2021,5.33,3.70
2022,2.92,1.65
2023,0.62,2.07
2024,0.60,1.45
2025,0.19,0.15


## Result 2 — SA-conversion overhead
Reconstruct in SA space (published SA components embed the harvested stratum factors,
`sa_floor.md` §5) vs the official SA aggregate. Overhead = SA MAE - NSA MAE.

In [5]:
rows = []
for agg in ['headline', 'core']:
    n = A.reconstruction_error(agg, seasonal='NSA')['mae_2023plus_bp']
    s = A.reconstruction_error(agg, seasonal='SA')['mae_2023plus_bp']
    rows.append({'aggregate': agg, 'NSA_MAE_2023+': n, 'SA_MAE_2023+': s, 'overhead_bp': round(s-n, 2)})
pd.DataFrame(rows).set_index('aggregate')

,NSA_MAE_2023+,SA_MAE_2023+,overhead_bp
aggregate,,,
headline,0.50,0.49,-0.01
core,1.32,1.36,0.04


**Findings.** Headline reconstructs to **0.50 bp (2023+)** — meets the ≤1 bp target under BLS's
current annual-weight methodology; the ~3 bp pre-2023 residual is the biennial-weight regime
(2021 surge), not a machinery error. Core is 1.32 bp (finer partition). **SA-conversion overhead
is ~0** (<0.05 bp) at the aggregate level — the SA pathway is effectively free for a good NSA
forecast, save the Jan/Feb annual factor seam. See `docs/aggregation_error.md`.